# Queen's GAMBIT — 2000+ ELO — 2026-04 dataset

Training signal: illegal move penalty added to cross-entropy loss.  
Data: Lichess 2026-04 (~89M games), filtered to 1800+ ELO, capped at 300,000 games.

**Runtime → T4 GPU before running.**

In [ ]:
!pip install chess matplotlib tqdm zstandard -q

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
open('src/__init__.py', 'w').close()

with open('src/vocab.py', 'w') as f:
    f.write('\nimport os\n\nGAME_START = "<|game_start|>"\nGAME_END   = "<|game_end|>"\nSPECIAL_TOKENS = [GAME_START, GAME_END]\n\ndef _enumerate_all_moves():\n    files, ranks, pieces = "abcdefgh", "12345678", "qrbn"\n    moves = []\n    for f1 in files:\n        for r1 in ranks:\n            for f2 in files:\n                for r2 in ranks:\n                    if f1+r1 != f2+r2:\n                        moves.append(f1+r1+f2+r2)\n    for f1 in files:\n        for f2 in files:\n            for p in pieces:\n                moves.append(f1+"7"+f2+"8"+p)\n                moves.append(f1+"2"+f2+"1"+p)\n    return moves\n\nclass ChessTokenizer:\n    def __init__(self):\n        vocab = SPECIAL_TOKENS + sorted(set(_enumerate_all_moves()))\n        self.move_to_id    = {m: i for i, m in enumerate(vocab)}\n        self.id_to_move    = {i: m for m, i in self.move_to_id.items()}\n        self.vocab_size    = len(vocab)\n        self.game_start_id = self.move_to_id[GAME_START]\n        self.game_end_id   = self.move_to_id[GAME_END]\n\n    def encode(self, moves): return [self.move_to_id[m] for m in moves]\n    def decode(self, ids):   return [self.id_to_move[i] for i in ids]\n\n    def save(self, path):\n        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n        with open(path, "w") as f:\n            for i in range(self.vocab_size):\n                f.write(self.id_to_move[i] + "\\\\n")\n        print(f"Vocab ({self.vocab_size} tokens) saved to {path}")\n')

with open('src/dataset.py', 'w') as f:
    f.write('\nimport torch, chess, chess.pgn, numpy as np\nfrom torch.utils.data import Dataset, DataLoader\nfrom src.vocab import ChessTokenizer\n\ndef _elo(v):\n    try: return int(v or 0)\n    except: return 0\n\ndef parse_pgn(path, tokenizer, max_games=None, min_elo=0):\n    from tqdm.auto import tqdm\n    games_data, skipped = [], 0\n    with open(path, encoding="utf-8", errors="ignore") as f:\n        pbar = tqdm(unit="games", desc="Parsing PGN")\n        while True:\n            game = chess.pgn.read_game(f)\n            if game is None: break\n            if min_elo > 0:\n                w = _elo(game.headers.get("WhiteElo", 0))\n                b = _elo(game.headers.get("BlackElo", 0))\n                if min(w, b) < min_elo:\n                    skipped += 1\n                    pbar.update()\n                    pbar.set_postfix(kept=len(games_data), skipped=skipped)\n                    continue\n            board  = chess.Board()\n            moves, legal_before = [], []\n            for move in game.mainline_moves():\n                legal_ids = [tokenizer.move_to_id[m.uci()]\n                             for m in board.legal_moves\n                             if m.uci() in tokenizer.move_to_id]\n                legal_before.append(legal_ids)\n                moves.append(move.uci())\n                board.push(move)\n            if moves:\n                games_data.append((moves, legal_before))\n            pbar.update()\n            pbar.set_postfix(kept=len(games_data), skipped=skipped)\n            if max_games and len(games_data) >= max_games:\n                break\n        pbar.close()\n    print(f"Loaded {len(games_data)} games (skipped {skipped} below ELO {min_elo})")\n    return games_data\n\ndef games_to_stream(games_data, tokenizer):\n    token_ids, all_legal = [], []\n    for moves, legal_before in games_data:\n        token_ids.append(tokenizer.game_start_id)\n        all_legal.append(legal_before[0] if legal_before else [])\n        for i, uci in enumerate(moves):\n            if uci in tokenizer.move_to_id:\n                token_ids.append(tokenizer.move_to_id[uci])\n                nxt = legal_before[i+1] if i+1 < len(legal_before) else []\n                all_legal.append(nxt)\n        token_ids.append(tokenizer.game_end_id)\n        all_legal.append([])\n    lengths = np.array([len(x) for x in all_legal], dtype=np.int32)\n    offsets = np.zeros(len(all_legal), dtype=np.int64)\n    if len(lengths) > 1:\n        offsets[1:] = np.cumsum(lengths[:-1])\n    flat = np.array([v for ids in all_legal for v in ids], dtype=np.int16)\n    return token_ids, (flat, lengths, offsets)\n\nclass ChessDataset(Dataset):\n    """Sliding window over [start_pos, end_pos) of the full token stream.\n    Absolute positions are used to index legal_data, so train/val share the same arrays safely."""\n    def __init__(self, token_ids, legal_data, start_pos, end_pos, context_length, stride, vocab_size):\n        flat, lengths, offsets = legal_data\n        self.flat, self.lengths, self.offsets = flat, lengths, offsets\n        self.ctx, self.vocab_size = context_length, vocab_size\n        self.seqs, self.targets, self.starts = [], [], []\n        for i in range(start_pos, end_pos - context_length, stride):\n            self.seqs.append(torch.tensor(token_ids[i:i+context_length]))\n            self.targets.append(torch.tensor(token_ids[i+1:i+context_length+1]))\n            self.starts.append(i + 1)  # absolute position in full stream\n\n    def __len__(self): return len(self.seqs)\n\n    def __getitem__(self, idx):\n        x, y = self.seqs[idx], self.targets[idx]\n        start = self.starts[idx]\n        mask = torch.zeros(self.ctx, self.vocab_size, dtype=torch.bool)\n        for j in range(self.ctx):\n            pos = start + j\n            if pos < len(self.lengths):\n                n = int(self.lengths[pos])\n                if n > 0:\n                    ids = self.flat[int(self.offsets[pos]):int(self.offsets[pos])+n].astype(np.int64)\n                    mask[j, ids] = True\n        return x, y, mask\n\ndef create_dataloader(token_ids, legal_data, start_pos, end_pos,\n                      context_length=256, stride=256,\n                      batch_size=64, shuffle=True, drop_last=True, num_workers=2, vocab_size=4546):\n    ds = ChessDataset(token_ids, legal_data, start_pos, end_pos, context_length, stride, vocab_size)\n    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,\n                      drop_last=drop_last, num_workers=num_workers, pin_memory=True)\n')

with open('src/model.py', 'w') as f:
    f.write('\nimport torch, torch.nn as nn\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(self, d_in, d_out, ctx, dropout, heads, qkv_bias=False):\n        super().__init__()\n        assert d_out % heads == 0\n        self.d_out, self.heads = d_out, heads\n        self.head_dim = d_out // heads\n        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)\n        self.proj = nn.Linear(d_out, d_out)\n        self.drop = nn.Dropout(dropout)\n        self.register_buffer("mask", torch.triu(torch.ones(ctx, ctx), diagonal=1))\n    def forward(self, x):\n        b, n, _ = x.shape\n        k = self.W_k(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        q = self.W_q(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        v = self.W_v(x).view(b,n,self.heads,self.head_dim).transpose(1,2)\n        s = q @ k.transpose(2,3)\n        s.masked_fill_(self.mask.bool()[:n,:n], -torch.inf)\n        w = self.drop(torch.softmax(s / k.shape[-1]**0.5, dim=-1))\n        return self.proj((w@v).transpose(1,2).contiguous().view(b,n,self.d_out))\n\nclass LayerNorm(nn.Module):\n    def __init__(self, d):\n        super().__init__()\n        self.scale = nn.Parameter(torch.ones(d))\n        self.shift = nn.Parameter(torch.zeros(d))\n    def forward(self, x):\n        m, v = x.mean(-1,keepdim=True), x.var(-1,keepdim=True,unbiased=False)\n        return self.scale*(x-m)/torch.sqrt(v+1e-5)+self.shift\n\nclass GELU(nn.Module):\n    def forward(self, x):\n        return 0.5*x*(1+torch.tanh((2/torch.pi)**0.5*(x+0.044715*x**3)))\n\nclass FeedForward(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),GELU(),nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]))\n    def forward(self, x): return self.net(x)\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.att = MultiHeadAttention(cfg["emb_dim"],cfg["emb_dim"],cfg["context_length"],cfg["drop_rate"],cfg["n_heads"],cfg["qkv_bias"])\n        self.ff   = FeedForward(cfg)\n        self.n1, self.n2 = LayerNorm(cfg["emb_dim"]), LayerNorm(cfg["emb_dim"])\n        self.drop = nn.Dropout(cfg["drop_rate"])\n    def forward(self, x):\n        x = x + self.drop(self.att(self.n1(x)))\n        x = x + self.drop(self.ff(self.n2(x)))\n        return x\n\nclass GPTModel(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])\n        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["emb_dim"])\n        self.drop     = nn.Dropout(cfg["drop_rate"])\n        self.blocks   = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])\n        self.norm     = LayerNorm(cfg["emb_dim"])\n        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)\n    def forward(self, idx):\n        b, t = idx.shape\n        x = self.tok_emb(idx) + self.pos_emb(torch.arange(t, device=idx.device))\n        return self.out_head(self.norm(self.blocks(self.drop(x))))\n')

with open('src/generate.py', 'w') as f:
    f.write('\nimport torch, chess\nfrom src.vocab import GAME_START\n\ndef _legal_ids(board, tokenizer):\n    return [tokenizer.move_to_id[m.uci()] for m in board.legal_moves if m.uci() in tokenizer.move_to_id]\n\ndef generate_move(model, board, ctx_ids, tokenizer, device, temperature=1.0):\n    model.eval()\n    ctx = model.pos_emb.weight.shape[0]\n    idx = torch.tensor(ctx_ids[-ctx:], dtype=torch.long, device=device).unsqueeze(0)\n    with torch.no_grad():\n        logits = model(idx)[0, -1, :]\n    legal = _legal_ids(board, tokenizer)\n    if not legal: raise ValueError("No legal moves")\n    mask = torch.full_like(logits, float("-inf"))\n    mask[legal] = logits[legal]\n    return tokenizer.id_to_move[torch.multinomial(torch.softmax(mask/temperature,-1),1).item()]\n\ndef generate_sample_game(model, tokenizer, device, num_moves=20, temperature=1.0):\n    board, ctx_ids, moves = chess.Board(), [tokenizer.game_start_id], []\n    for _ in range(num_moves):\n        if board.is_game_over(): break\n        try:\n            uci = generate_move(model, board, ctx_ids, tokenizer, device, temperature)\n        except: break\n        board.push(chess.Move.from_uci(uci))\n        ctx_ids.append(tokenizer.move_to_id[uci])\n        moves.append(uci)\n    return moves\n')

print('Source files written.')


## Config

In [ ]:
import torch
from src.vocab import ChessTokenizer

tokenizer = ChessTokenizer()
print(f'Vocab size: {tokenizer.vocab_size}')
tokenizer.save('data/vocab.txt')

CHESS_GPT_CONFIG = {
    'vocab_size':     tokenizer.vocab_size,
    'context_length': 256,
    'emb_dim':        256,
    'n_heads':        8,
    'n_layers':       6,
    'drop_rate':      0.1,
    'qkv_bias':       False,
}

TRAIN_SETTINGS = {
    'learning_rate':    3e-4,
    'weight_decay':     0.1,
    'num_epochs':       10,
    'batch_size':       128,
    'eval_freq':        200,
    'eval_iter':        10,
    'min_elo':          2000,
    'max_games':        500000,
    'illegal_lambda':   0.1,   # weight of illegal move penalty in loss
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Download Lichess 2026-04

In [ ]:
import os, zstandard, requests

PGN_PATH = 'data/lichess_2026-04.pgn'
URL      = 'https://database.lichess.org/standard/lichess_db_standard_rated_2026-04.pgn.zst'

def download_lichess(url, pgn_path):
    zst = pgn_path + '.zst'
    headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'}
    r = requests.get(url, headers=headers, stream=True, timeout=600)
    print(f'HTTP {r.status_code}  Size: {r.headers.get("Content-Length","?")}')
    if r.status_code != 200: raise RuntimeError(f'HTTP {r.status_code}')
    total, received = int(r.headers.get('Content-Length', 0)), 0
    with open(zst, 'wb') as f:
        for chunk in r.iter_content(1<<20):
            f.write(chunk); received += len(chunk)
            if total: print(f'  {received/1e6:.0f}/{total/1e6:.0f} MB', end='\r')
    print(f'\nDecompressing...')
    with open(zst,'rb') as s, open(pgn_path,'wb') as d:
        zstandard.ZstdDecompressor().copy_stream(s, d)
    os.remove(zst)
    print(f'Done: {os.path.getsize(pgn_path)/1e6:.0f} MB')

if not os.path.exists(PGN_PATH):
    download_lichess(URL, PGN_PATH)
else:
    print(f'Already exists: {os.path.getsize(PGN_PATH)/1e6:.0f} MB')


## Load + tokenize (with legal move masks)

In [ ]:
from src.dataset import parse_pgn, games_to_stream, create_dataloader

games_data = parse_pgn(
    PGN_PATH, tokenizer,
    max_games=TRAIN_SETTINGS['max_games'],
    min_elo=TRAIN_SETTINGS['min_elo'],
)

token_ids, legal_data = games_to_stream(games_data, tokenizer)
print(f'Total tokens: {{len(token_ids):,}}')
del games_data  # free memory

split   = int(0.9 * len(token_ids))
ctx, bs = CHESS_GPT_CONFIG['context_length'], TRAIN_SETTINGS['batch_size']

train_loader = create_dataloader(token_ids, legal_data, 0, split, ctx, ctx // 2, bs,
                                  shuffle=True,  drop_last=True,
                                  vocab_size=tokenizer.vocab_size)
val_loader   = create_dataloader(token_ids, legal_data, split, len(token_ids), ctx, ctx, bs,
                                  shuffle=False, drop_last=False,
                                  vocab_size=tokenizer.vocab_size)
print(f'Train batches: {{len(train_loader)}} | Val batches: {{len(val_loader)}}')


## Build model

In [ ]:
from src.model import GPTModel
import torch

torch.manual_seed(42)
model = GPTModel(CHESS_GPT_CONFIG).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


## Train

In [ ]:
import torch.nn as nn
from src.generate import generate_sample_game

def calc_loss_batch(x, y, legal_mask, model, device, illegal_lambda=0.1):
    logits = model(x.to(device))                          # (B, T, V)
    ce_loss = nn.functional.cross_entropy(
        logits.flatten(0,1), y.to(device).flatten())
    probs         = torch.softmax(logits, dim=-1)
    legal         = legal_mask.to(device)
    has_mask      = legal.any(dim=-1, keepdim=True)
    illegal_probs = probs * (~legal).float()
    penalty = (illegal_probs.sum(-1) * has_mask.squeeze(-1).float()).mean()
    return ce_loss + illegal_lambda * penalty

def calc_loss_loader(loader, model, device, num_batches=None):
    total, n = 0.0, 0
    nb = min(num_batches or len(loader), len(loader))
    for i, (x, y, lm) in enumerate(loader):
        if i >= nb: break
        total += calc_loss_batch(x, y, lm, model, device).item(); n += 1
    return total/n if n else float('nan')

def train(model, train_loader, val_loader, optimizer, scheduler, device,
          num_epochs, eval_freq, eval_iter, tokenizer, illegal_lambda):
    train_losses, val_losses, tokens_seen_list = [], [], []
    tokens_seen, step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for x, y, lm in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(x, y, lm, model, device, illegal_lambda)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tokens_seen += x.numel(); step += 1
            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    tl = calc_loss_loader(train_loader, model, device, eval_iter)
                    vl = calc_loss_loader(val_loader,   model, device, eval_iter)
                model.train()
                train_losses.append(tl); val_losses.append(vl)
                tokens_seen_list.append(tokens_seen)
                lr = optimizer.param_groups[0]['lr']
                print(f'Ep {epoch+1} (Step {step:06d}): Train {tl:.3f}  Val {vl:.3f}  LR {lr:.2e}')
        scheduler.step()
        sample = generate_sample_game(model, tokenizer, device, num_moves=10)
        print(f'  Sample: {" ".join(sample)}')
        ckpt = f'checkpoints/chess_gpt_ep{epoch+1}.pth'
        torch.save(model.state_dict(), ckpt)
        print(f'  Saved: {ckpt}')
    return train_losses, val_losses, tokens_seen_list

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_SETTINGS['learning_rate'],
    weight_decay=TRAIN_SETTINGS['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_SETTINGS['num_epochs'])

train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs=TRAIN_SETTINGS['num_epochs'],
    eval_freq=TRAIN_SETTINGS['eval_freq'],
    eval_iter=TRAIN_SETTINGS['eval_iter'],
    tokenizer=tokenizer,
    illegal_lambda=TRAIN_SETTINGS['illegal_lambda'],
)


## Loss plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,4))
steps = list(range(len(train_losses)))
ax.plot(steps, train_losses, label='Train'); ax.plot(steps, val_losses, linestyle='-.', label='Val')
ax.set_xlabel('Eval step'); ax.set_ylabel('Loss')
ax.set_title(f'Queen's GAMBIT {TRAIN_SETTINGS["min_elo"]}+ ELO'); ax.legend()
plt.tight_layout(); plt.savefig('loss.png', dpi=150); plt.show()
print(f'Final  train: {train_losses[-1]:.3f}  val: {val_losses[-1]:.3f}')


## Watch model play

In [ ]:
import chess
from src.generate import generate_move

model.eval()
board, ctx_ids, move_num = chess.Board(), [tokenizer.game_start_id], 1

for _ in range(400):
    if board.is_game_over(): break
    is_white = board.turn == chess.WHITE
    uci  = generate_move(model, board, ctx_ids, tokenizer, device, temperature=0.8)
    move = chess.Move.from_uci(uci)
    san  = board.san(move)
    board.push(move)
    ctx_ids.append(tokenizer.move_to_id[uci])
    if is_white: print(f'{move_num:3}. {san}', end='')
    else:        print(f'  {san}'); move_num += 1

print(f'\nResult: {board.result()}')
if board.is_checkmate():    print('CHECKMATE')
elif board.is_stalemate():  print('STALEMATE')
elif board.is_repetition(): print('DRAW - repetition')
elif board.is_fifty_moves():print('DRAW - 50 move rule')
print(board)


## Save to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp checkpoints/chess_gpt_ep10.pth /content/drive/MyDrive/
